# Summary

Explore the RAG evaluation datasets

In [1]:
import os, sys
import pandas as pd
import json
import time

# AWS Python
import boto3
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv("../.env")

# Configuration
region_name    = os.getenv("AWS_REGION", "us-east-1")
profile_name   = os.getenv("AWS_PROFILE", "default")
bucket         = os.getenv("S3_BUCKET", "rag-search-tests")
s3_prefix      = os.getenv("S3_PREFIX", "documents/")
collection_name = os.getenv("COLLECTION_NAME", "rag-kb-collection")
kb_name        = os.getenv("KB_NAME", "rag-search-kb")
index_name     = os.getenv("INDEX_NAME", "bedrock-knowledge-base-default-index")
role_name      = os.getenv("ROLE_NAME", "AmazonBedrockExecutionRoleForKnowledgeBase")
embedding_model = os.getenv("EMBEDDING_MODEL", "amazon.titan-embed-text-v2:0")
kb_id          = os.getenv("KB_ID")

## Create a Bedrock Knowledge Base

In [2]:
"""
Complete Guide to Creating an Amazon Bedrock Knowledge Base
============================================================

Prerequisites:
1. S3 bucket with .txt files containing document text
2. AWS credentials configured
3. Python packages: boto3, opensearchpy

This script assumes .txt files are already uploaded to S3
"""


'\nComplete Guide to Creating an Amazon Bedrock Knowledge Base\n============================================================\n\nPrerequisites:\n1. S3 bucket with .txt files containing document text\n2. AWS credentials configured\n3. Python packages: boto3, opensearchpy\n\nThis script assumes .txt files are already uploaded to S3\n'

## Set up configuration and initialize

In [2]:
# Use Admin for all Setup tasks to avoid AccessDenied
admin_session = boto3.Session(profile_name=profile_name,
                              region_name=region_name)

aoss_client   = admin_session.client('opensearchserverless', region_name=region_name)
iam_client    = admin_session.client('iam')
sts_client    = admin_session.client('sts')
bedrock_agent = admin_session.client('bedrock-agent', region_name=region_name)

account_id       = sts_client.get_caller_identity()['Account']
current_user_arn = sts_client.get_caller_identity()['Arn']

## Create vector index

In [ ]:
# # Set up OpenSearch client with AWS auth using admin_session
# credentials = admin_session.get_credentials()
# awsauth = AWSV4SignerAuth(credentials, region_name, 'aoss')

# # Get the collection endpoint
# collection_endpoint = aoss_client.batch_get_collection(names=[collection_name])['collectionDetails'][0][
#     'collectionEndpoint']
# host = collection_endpoint.replace('https://', '')

# os_client = OpenSearch(
#     hosts=[{'host': host, 'port': 443}],
#     http_auth=awsauth,
#     use_ssl=True,
#     verify_certs=True,
#     connection_class=RequestsHttpConnection,
#     timeout=300
# )

# # Create the index with proper mapping for Bedrock
# index_body = {
#     "settings": {
#         "index.knn": True
#     },
#     "mappings": {
#         "properties": {
#             "bedrock-knowledge-base-default-vector": {
#                 "type": "knn_vector",
#                 "dimension": 1024,
#                 "method": {
#                     "engine": "faiss",
#                     "name": "hnsw"
#                 }
#             },
#             "AMAZON_BEDROCK_TEXT_CHUNK": {
#                 "type": "text"
#             },
#             "AMAZON_BEDROCK_METADATA": {
#                 "type": "text"
#             }
#         }
#     }
# }

# # Create index if it doesn't exist
# if not os_client.indices.exists(index=index_name):
#     os_client.indices.create(index=index_name, body=index_body)
#     print(f"✓ Vector index created: {index_name}")
# else:
#     print(f"✓ Vector index already exists: {index_name}. To delete, use os_client.indices.delete(index=index_name)")
#     # os_client.indices.delete(index=index_name)

# # Wait for index to be ready
# time.sleep(5)

## Set up Knowledge Base

In [ ]:
# DISABLED: Only run this cell if you need to delete and recreate the Knowledge Base from scratch.
# When using a pre-existing KB (kb_id set in .env), skip this cell entirely.

# # --- Create OpenSearch Serverless Security Policies and Bedrock Role ---
# def ensure_aoss_policy(name, policy_type, policy_doc):
#     try:
#         aoss_client.create_security_policy(name=name, type=policy_type, policy=json.dumps(policy_doc))
#         print(f"Created {policy_type} policy: {name}")
#     except Exception as e:
#         if 'ConflictException' in str(e):
#             print(f"{policy_type.capitalize()} policy already exists: {name}")
#         else:
#             raise


# def ensure_execution_role():
#     trust_pol = {
#         "Version": "2012-10-17",
#         "Statement": [
#             {
#                 "Effect": "Allow",
#                 "Principal": {"Service": "bedrock.amazonaws.com"},
#                 "Action": "sts:AssumeRole"
#             }
#         ]
#     }

#     try:
#         role_arn = iam_client.create_role(
#             RoleName=role_name,
#             AssumeRolePolicyDocument=json.dumps(trust_pol)
#         )['Role']['Arn']
#         print(f"Created role: {role_name}")
#     except iam_client.exceptions.EntityAlreadyExistsException:
#         role_arn = iam_client.get_role(RoleName=role_name)['Role']['Arn']
#         print(f"Role already exists: {role_name}")

#     exec_pol = {
#         "Version": "2012-10-17",
#         "Statement": [
#             {
#                 "Effect": "Allow",
#                 "Action": ["s3:GetObject", "s3:ListBucket"],
#                 "Resource": [f"arn:aws:s3:::{bucket}", f"arn:aws:s3:::{bucket}/*"]
#             },
#             {
#                 "Effect": "Allow",
#                 "Action": ["aoss:APIAccessAll"],
#                 "Resource": ["*"]
#             },
#             {
#                 "Effect": "Allow",
#                 "Action": ["bedrock:InvokeModel"],
#                 "Resource": [f"arn:aws:bedrock:{region_name}::foundation-model/{embedding_model}"]
#             }
#         ]
#     }

#     # Always upsert policy to avoid stale role permissions.
#     iam_client.put_role_policy(
#         RoleName=role_name,
#         PolicyName='BedrockPolicy',
#         PolicyDocument=json.dumps(exec_pol)
#     )
#     print("Upserted inline role policy: BedrockPolicy")
#     return role_arn


# def ensure_data_access_policy(role_arn):
#     policy_name = f"{collection_name}-acc"
#     rn = role_arn.split('/')[-1]

#     access_policy = [{
#         "Rules": [
#             {
#                 "Resource": [f"collection/{collection_name}"],
#                 "Permission": ["aoss:*"],
#                 "ResourceType": "collection"
#             },
#             {
#                 "Resource": [f"index/{collection_name}/*"],
#                 "Permission": ["aoss:*"],
#                 "ResourceType": "index"
#             }
#         ],
#         "Principal": [
#             role_arn,
#             f"arn:aws:sts::{account_id}:assumed-role/{rn}/*",
#             current_user_arn
#         ]
#     }]

#     try:
#         aoss_client.create_access_policy(
#             name=policy_name,
#             type='data',
#             policy=json.dumps(access_policy)
#         )
#         print(f"Created data access policy: {policy_name}")
#     except Exception as e:
#         if 'ConflictException' in str(e):
#             existing_policy = aoss_client.get_access_policy(name=policy_name, type='data')
#             try:
#                 aoss_client.update_access_policy(
#                     name=policy_name,
#                     type='data',
#                     policyVersion=existing_policy['accessPolicyDetail']['policyVersion'],
#                     policy=json.dumps(access_policy)
#                 )
#                 print(f"Updated data access policy: {policy_name}")
#             except Exception as ve:
#                 if 'No changes detected' in str(ve):
#                     print(f"Data access policy already up to date: {policy_name}")
#                 else:
#                     raise
#         else:
#             raise


# ensure_aoss_policy(
#     name=f'{collection_name}-enc',
#     policy_type='encryption',
#     policy_doc={
#         "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{collection_name}"]}],
#         "AWSOwnedKey": True
#     }
# )

# ensure_aoss_policy(
#     name=f'{collection_name}-net',
#     policy_type='network',
#     policy_doc=[{
#         "Rules": [
#             {"ResourceType": "collection", "Resource": [f"collection/{collection_name}"]},
#             {"ResourceType": "dashboard", "Resource": [f"collection/{collection_name}"]}
#         ],
#         "AllowFromPublic": True
#     }]
# )

# role_arn = ensure_execution_role()
# ensure_data_access_policy(role_arn)
# print("✓ Security roles and policies established.")
# print(f"Checking for existing Knowledge Base named '{kb_name}'...")
# existing_kb_id = None
# paginator = bedrock_agent.get_paginator('list_knowledge_bases')
# for page in paginator.paginate():
#     for kb in page['knowledgeBaseSummaries']:
#         if kb['name'] == kb_name:
#             existing_kb_id = kb['knowledgeBaseId']
#             print(f"Found existing KB {existing_kb_id}. Deleting...")
#             bedrock_agent.delete_knowledge_base(knowledgeBaseId=existing_kb_id)
#
# if existing_kb_id:
#     start = time.time()
#     while True:
#         kb_ids = []
#         paginator = bedrock_agent.get_paginator('list_knowledge_bases')
#         for page in paginator.paginate():
#             kb_ids.extend([k['knowledgeBaseId'] for k in page['knowledgeBaseSummaries']])
#
#         if existing_kb_id not in kb_ids:
#             print("✓ Existing KB deleted")
#             break
#
#         if time.time() - start > 180:
#             raise TimeoutError(f"Timed out waiting for KB {existing_kb_id} deletion")
#
#         print("Waiting for KB deletion to complete...")
#         time.sleep(10)

In [ ]:
# DISABLED: Only run this cell if you need to provision a brand new Knowledge Base.
# When using a pre-existing KB (kb_id set in .env), skip this cell entirely.

# # --- Create OpenSearch Collection ---
# try:
#     coll = aoss_client.create_collection(name=collection_name, type='VECTORSEARCH')
#     coll_arn = coll['createCollectionDetail']['arn']
#     print("Waiting for collection to activate...")
# except Exception:
#     coll_arn = aoss_client.batch_get_collection(names=[collection_name])['collectionDetails'][0]['arn']
#
# while aoss_client.batch_get_collection(names=[collection_name])['collectionDetails'][0]['status'] != 'ACTIVE':
#     time.sleep(5)
#
# # --- Create Knowledge Base (retry for eventual consistency) ---
# last_err = None
# for attempt in range(1, 8):
#     try:
#         kb_response = bedrock_agent.create_knowledge_base(
#             name=kb_name,
#             roleArn=role_arn,
#             knowledgeBaseConfiguration={
#                 'type': 'VECTOR',
#                 'vectorKnowledgeBaseConfiguration': {
#                     'embeddingModelArn': f'arn:aws:bedrock:{region_name}::foundation-model/{embedding_model}'
#                 }
#             },
#             storageConfiguration={
#                 'type': 'OPENSEARCH_SERVERLESS',
#                 'opensearchServerlessConfiguration': {
#                     'collectionArn': coll_arn,
#                     'vectorIndexName': index_name,
#                     'fieldMapping': {
#                         'vectorField': 'bedrock-knowledge-base-default-vector',
#                         'textField': 'AMAZON_BEDROCK_TEXT_CHUNK',
#                         'metadataField': 'AMAZON_BEDROCK_METADATA'
#                     }
#                 }
#             }
#         )
#         kb_id = kb_response['knowledgeBase']['knowledgeBaseId']
#         print(f"✓ Knowledge Base Created: {kb_id}")
#         break
#     except bedrock_agent.exceptions.ValidationException as e:
#         last_err = e
#         if 'security_exception' in str(e) and attempt < 7:
#             wait_seconds = attempt * 15
#             print(f"Attempt {attempt}/7 failed due to AOSS security propagation. Retrying in {wait_seconds}s...")
#             time.sleep(wait_seconds)
#             continue
#         raise
#
# if 'kb_id' not in locals():
#     raise RuntimeError(f"Failed to create knowledge base after retries: {last_err}")

## Create Security Policies

In [3]:
# --- Retrieve the execution role from the existing Knowledge Base ---
# Instead of configuring our own role, use whatever role the KB was created with.
# That is the role Bedrock will actually use when ingesting documents and accessing S3.

kb_meta = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)
role_arn  = kb_meta['knowledgeBase']['roleArn']
role_name = role_arn.split('/')[-1]

print(f"KB {kb_id} uses execution role:")
print(f"  Name: {role_name}")
print(f"  ARN:  {role_arn}")

KB TRVACJAJWE uses execution role:
  Name: amplify-coriagent-johnhal-CORIBedrockKnowledgeBaseR-rY31IF2EDSnA
  ARN:  arn:aws:iam::312512371189:role/amplify-coriagent-johnhal-CORIBedrockKnowledgeBaseR-rY31IF2EDSnA


## Check permissions

In [6]:
# Add before creating KB
print(f"Using role: {role_arn}")
try:
    rn = role_arn.split('/')[-1]
    iam_client.get_role(RoleName=rn)
    print("✓ Role exists")
except Exception as e:
    print(f"✗ Role issue: {e}")

Using role: arn:aws:iam::312512371189:role/amplify-coriagent-johnhal-CORIBedrockKnowledgeBaseR-rY31IF2EDSnA
✓ Role exists


## Add security roles

In [7]:

# Wait for IAM and AOSS policy propagation before KB creation
MAX_WAIT_SECONDS = 180
SLEEP_SECONDS = 10

print(f"Using role: {role_arn}")
iam_client.get_role(RoleName=role_name)

start = time.time()
while True:
    elapsed = time.time() - start
    if elapsed > MAX_WAIT_SECONDS:
        raise TimeoutError("Timed out waiting for IAM/AOSS policy propagation.")

    try:
        pol = aoss_client.get_access_policy(name=f"{collection_name}-acc", type='data')
        version = pol['accessPolicyDetail']['policyVersion']
        print(f"✓ Access policy visible (version {version}); proceeding.")
        break
    except Exception as e:
        print(f"Waiting for policy propagation ({int(elapsed)}s): {e}")
        time.sleep(SLEEP_SECONDS)

Using role: arn:aws:iam::312512371189:role/amplify-coriagent-johnhal-CORIBedrockKnowledgeBaseR-rY31IF2EDSnA
✓ Access policy visible (version MTc3NzM5NzIyOTQyMl8x); proceeding.


## Inspect KB storage and patch RDS schema

Bedrock KB `TRVACJAJWE` is backed by Amazon RDS (Aurora PostgreSQL). Each metadata attribute we attach in S10 must exist as a column in the embeddings table or ingestion fails with `column does not exist`.

This cell first prints the KB's storage configuration so you can see the cluster ARN, database, table name, and field mapping. Then it adds any missing metadata columns to the table via the RDS Data API.

In [8]:
# ============================================================================
# Inspect KB storage configuration and patch RDS schema
# ============================================================================
# Step 1 (option 2): print the KB's storage configuration so we can see exactly
# how Bedrock is wired to the vector backend (RDS table, columns, secret, etc.)

storage = kb_meta['knowledgeBase']['storageConfiguration']
print("=" * 60)
print("KB Storage Configuration")
print("=" * 60)
print(json.dumps(storage, indent=2, default=str))

# Extract RDS connection details for the schema patch below.
rds_config  = storage.get('rdsConfiguration', {})
cluster_arn = rds_config.get('resourceArn')
secret_arn  = rds_config.get('credentialsSecretArn')
db_name     = rds_config.get('databaseName')
table_name  = rds_config.get('tableName')

# Step 2 (option 1): add the metadata columns Bedrock expects to write into.
# Each metadata attribute we set in S10 (`source_url`, `source_type`,
# `document_index`) becomes a column on the embeddings table during ingestion.
# If the column doesn't exist, the RDS write fails with SQLState 42703.

print("\n" + "=" * 60)
print(f"Adding metadata columns to {table_name}")
print("=" * 60)

rds_data = admin_session.client('rds-data', region_name=region_name)
required_columns = ['source_url', 'source_type', 'document_index']

for col in required_columns:
    sql = f'ALTER TABLE {table_name} ADD COLUMN IF NOT EXISTS {col} TEXT'
    try:
        rds_data.execute_statement(
            resourceArn=cluster_arn,
            secretArn=secret_arn,
            database=db_name,
            sql=sql
        )
        print(f"  ✓ {col}")
    except Exception as e:
        print(f"  ✗ {col}: {e}")

KB Storage Configuration
{
  "type": "RDS",
  "rdsConfiguration": {
    "resourceArn": "arn:aws:rds:us-east-1:312512371189:cluster:amplify-coriagent-johnhal-coriknowledgebasecluster-ejpfecxuges5",
    "credentialsSecretArn": "arn:aws:secretsmanager:us-east-1:312512371189:secret:amplifycoriagentjohnhallsan-RQPP85v8AVBa-OVrbQc",
    "databaseName": "cori_agent_data",
    "tableName": "cori_agent_embeddings",
    "fieldMapping": {
      "primaryKeyField": "id",
      "vectorField": "embedding",
      "textField": "text",
      "metadataField": "metadata"
    }
  }
}

Adding metadata columns to cori_agent_embeddings
  ✓ source_url
  ✓ source_type
  ✓ document_index


## Ingest documents to the Knowledge Base

In [9]:

# ============================================================================
# SECTION 3: Ingest Source Data into existing Knowledge Base
# ============================================================================
print(f"Ingesting into Knowledge Base: {kb_id}")
print(f"Source: s3://{bucket}/{s3_prefix}")

# --- Add S3 Data Source ---
s3_config = {
    'bucketArn': f'arn:aws:s3:::{bucket}'
}
if s3_prefix:
    s3_config['inclusionPrefixes'] = [s3_prefix]

# Use stable naming to allow reruns.
existing_data_source = None
paginator = bedrock_agent.get_paginator('list_data_sources')
for page in paginator.paginate(knowledgeBaseId=kb_id):
    for ds in page['dataSourceSummaries']:
        if ds['name'] == 's3-docs-with-metadata':
            existing_data_source = ds['dataSourceId']
            break

if existing_data_source:
    ds_id = existing_data_source
    print(f"Reusing existing data source: {ds_id}")
else:
    ds_response = bedrock_agent.create_data_source(
        knowledgeBaseId=kb_id,
        name='s3-docs-with-metadata',
        dataSourceConfiguration={
            'type': 'S3',
            's3Configuration': s3_config
        }
    )
    ds_id = ds_response['dataSource']['dataSourceId']
    print(f"Created data source: {ds_id}")

# --- Start Ingestion ---
ingestion_response = bedrock_agent.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
print(f"✓ Ingestion started for {bucket}/{s3_prefix}")
print("You can now filter by 'source_type' or 'doc_index' in your queries!")

Ingesting into Knowledge Base: TRVACJAJWE
Source: s3://cori.agent.kb/dev/knowledge_base/docs/rag_search/
Reusing existing data source: EJBPSLLHJ3
✓ Ingestion started for cori.agent.kb/dev/knowledge_base/docs/rag_search/
You can now filter by 'source_type' or 'doc_index' in your queries!


## Check status of ingestion job

In [10]:
ingestion_job_id = ingestion_response['ingestionJob']['ingestionJobId']
print(f"Monitoring Ingestion Job: {ingestion_job_id}...")

while True:
    job_response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=ds_id,
        ingestionJobId=ingestion_job_id
    )

    status = job_response['ingestionJob']['status']
    print(f"Current Status: {status}")

    if status in ['COMPLETE', 'FAILED', 'STOPPED']:
        break

    time.sleep(10)

# --- Detailed Reporting ---
job_data = job_response['ingestionJob']

print("\n" + "="*30)
print("INGESTION JOB SUMMARY")
print("="*30)
print(f"Status: {job_data['status']}")

# Print Statistics if available
if 'statistics' in job_data:
    stats = job_data['statistics']
    print(f"Documents Scanned: {stats.get('numberOfDocumentsScanned', 0)}")
    print(f"Documents Indexed: {stats.get('numberOfNewDocumentsIndexed', 0)}")
    print(f"Documents Failed:  {stats.get('numberOfDocumentsFailed', 0)}")

# Print Failure Reasons
if 'failureReasons' in job_data:
    print("\nFailure Reasons:")
    for reason in job_data['failureReasons']:
        print(f"  - {reason}")

# Full Debugging Dump
print("\nFull JSON Job Details (for debugging):")
print(json.dumps(job_data, indent=2, default=str))


Monitoring Ingestion Job: YZAAAMRVGQ...
Current Status: IN_PROGRESS
Current Status: IN_PROGRESS
Current Status: IN_PROGRESS
Current Status: COMPLETE

INGESTION JOB SUMMARY
Status: COMPLETE
Documents Scanned: 20
Documents Indexed: 0
Documents Failed:  0

Full JSON Job Details (for debugging):
{
  "knowledgeBaseId": "TRVACJAJWE",
  "dataSourceId": "EJBPSLLHJ3",
  "ingestionJobId": "YZAAAMRVGQ",
  "status": "COMPLETE",
  "statistics": {
    "numberOfDocumentsScanned": 20,
    "numberOfMetadataDocumentsScanned": 20,
    "numberOfNewDocumentsIndexed": 0,
    "numberOfModifiedDocumentsIndexed": 20,
    "numberOfMetadataDocumentsModified": 0,
    "numberOfDocumentsDeleted": 0,
    "numberOfDocumentsFailed": 0
  },
  "startedAt": "2026-04-28 19:08:02.619270+00:00",
  "updatedAt": "2026-04-28 19:08:35.796535+00:00"
}
